# Module 1: Data Ingestion

## Overview

This notebook implements the **Data Ingestion** module of the **TrustGuard: Data Quality Pipeline for Retail Data** project.

The objective of this module is to load the raw retail transaction dataset into Databricks, validate the dataset schema, identify missing or unexpected columns, standardize column names, and store the validated dataset in the Raw Layer as a Delta Table for further processing.

In [0]:
# Read CSV file from Databricks Volume

df = spark.read.csv(
    "/Volumes/workspace/default/trustguard_data/retail_transactions.csv",
    header=True,
    inferSchema=True
)

display(df.limit(10))

Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,true
TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,true
TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,false
TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,null
TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,false
TXN_7482416,CUST_09,Patisserie,null,null,10.0,200.0,Credit Card,Online,2023-11-30,null
TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,true
TXN_1372952,CUST_21,Furniture,null,33.5,null,null,Digital Wallet,In-store,2024-04-02,true
TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,false
TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,false


In [0]:
# Count total records and total columns

print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 12575
Total Columns: 11


In [0]:
# Print all column names and schema 

print(df.columns)

df.printSchema()

['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']
root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Total Spent: double (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: date (nullable = true)
 |-- Discount Applied: boolean (nullable = true)



In [0]:
# Expected columns according to problem statement

expected_columns = [
    "Transaction ID",
    "Customer ID",
    "Category",
    "Item",
    "Price Per Unit",
    "Quantity",
    "Total Spent",
    "Payment Method",
    "Location",
    "Transaction Date",
    "Discount Applied"
]

actual_columns=df.columns

missing_columns=[
col
for col in expected_columns
if col not in actual_columns
]

unexpected_columns=[
col
for col in actual_columns
if col not in expected_columns
]

print("Missing Columns :",missing_columns)

print("Unexpected Columns :",unexpected_columns)

Missing Columns : []
Unexpected Columns : []


In [0]:
# Stop pipeline if required columns are missing

if missing_columns:
    raise Exception(f"Pipeline Stopped! Missing Columns: {missing_columns}")
else:
    print("Schema Validation Passed")

Schema Validation Passed


In [0]:
# Standardize Column Names
# Convert column names into lowercase snake_case format

import re

clean_columns = [
    re.sub(r"[^a-zA-Z0-9]+", "_", c.strip().lower()).strip("_")
    for c in df.columns
]

raw_df = df.toDF(*clean_columns)

print(raw_df.columns)

display(raw_df.limit(10))

['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit', 'quantity', 'total_spent', 'payment_method', 'location', 'transaction_date', 'discount_applied']


transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,true
TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,true
TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,false
TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,null
TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,false
TXN_7482416,CUST_09,Patisserie,null,null,10.0,200.0,Credit Card,Online,2023-11-30,null
TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,true
TXN_1372952,CUST_21,Furniture,null,33.5,null,null,Digital Wallet,In-store,2024-04-02,true
TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,false
TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,false


In [0]:
# Create Schema
spark.sql("CREATE SCHEMA IF NOT EXISTS trustguard")

# Save Raw Layer as Delta Table
raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("trustguard.raw_transactions")

In [0]:
# Save standardized Raw DataFrame as Delta Table

raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("trustguard.raw_transactions")

print("Raw Delta Table Created Successfully.")

Raw Delta Table Created Successfully.


In [0]:
# Verify Raw Delta Table

display(
    spark.table("trustguard.raw_transactions").limit(10)
)

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,true
TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,true
TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,false
TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,null
TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,false
TXN_7482416,CUST_09,Patisserie,null,null,10.0,200.0,Credit Card,Online,2023-11-30,null
TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,true
TXN_1372952,CUST_21,Furniture,null,33.5,null,null,Digital Wallet,In-store,2024-04-02,true
TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,false
TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,false
